<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/switchingtoH3grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install required enterprise libraries
!pip install -q openmeteo-requests requests-cache retry-requests xarray netCDF4

import numpy as np
from google.colab import files

# Nalbari Exact Bounding Box
lat_min, lat_max = 26.14, 26.59
lon_min, lon_max = 91.24, 91.64
resolution = 0.05

# Generate the grid arrays
lats = np.arange(lat_min, lat_max + resolution, resolution)
lons = np.arange(lon_min, lon_max + resolution, resolution)

# Create flattened coordinate lists
lat_list = [round(lat, 3) for lat in lats for lon in lons]
lon_list = [round(lon, 3) for lat in lats for lon in lons]

print(f"Generated {len(lat_list)} spatial extraction points across Nalbari.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.5/772.5 kB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.0/396.0 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 95.6 MB/s eta 0:00:00
Generated 100 spatial extraction points across Nalbari.


In [4]:
import openmeteo_requests
import requests_cache
from retry_requests import retry
import pandas as pd
import xarray as xr

# Setup Open-Meteo client
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat_list,
    "longitude": lon_list,
    "start_date": "2021-01-01",
    "end_date": "2023-12-31",
    "daily": ["temperature_2m_max", "temperature_2m_mean", "precipitation_sum", "et0_fao_evapotranspiration"],
    "timezone": "Asia/Kolkata"
}

print("Initiating batch API extraction. This may take 30-60 seconds...")
responses = openmeteo.weather_api(url, params=params)
print("Data extracted. Compiling into Data Cube...")

dfs = []
for i, response in enumerate(responses):
    daily = response.Daily()

    # FIX: Added .tz_localize(None) to strip UTC metadata for NetCDF compatibility
    time_index = pd.date_range(
        start=pd.to_datetime(daily.Time(), unit="s", utc=True),
        end=pd.to_datetime(daily.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=daily.Interval()),
        inclusive="left"
    ).tz_localize(None)

    df = pd.DataFrame({
        "time": time_index.normalize(),
        "lat": lat_list[i],
        "lon": lon_list[i],
        "temp_max": daily.Variables(0).ValuesAsNumpy(),
        "temp_mean": daily.Variables(1).ValuesAsNumpy(),
        "precip": daily.Variables(2).ValuesAsNumpy(),
        "evapotranspiration": daily.Variables(3).ValuesAsNumpy()
    })
    dfs.append(df)

# Build Xarray Dataset
master_df = pd.concat(dfs)
master_df.set_index(['time', 'lat', 'lon'], inplace=True)
ds = master_df.to_xarray()

# Save locally to Colab's temporary storage, then trigger browser download
file_name = "nalbari_weather_cube_2021_2023.nc"
ds.to_netcdf(file_name)
print(f"Data Cube built successfully. Triggering local download for {file_name}...")

# Force browser download
files.download(file_name)

# Print the structure for verification
print("\n--- DATA CUBE STRUCTURE ---")
print(ds)

Initiating batch API extraction. This may take 30-60 seconds...
Data extracted. Compiling into Data Cube...
Data Cube built successfully. Triggering local download for nalbari_weather_cube_2021_2023.nc...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- DATA CUBE STRUCTURE ---
<xarray.Dataset> Size: 2MB
Dimensions:             (time: 1095, lat: 10, lon: 10)
Coordinates:
  * time                (time) datetime64[ns] 9kB 2020-12-31 ... 2023-12-30
  * lat                 (lat) float64 80B 26.14 26.19 26.24 ... 26.54 26.59
  * lon                 (lon) float64 80B 91.24 91.29 91.34 ... 91.64 91.69
Data variables:
    temp_max            (time, lat, lon) float32 438kB 23.1 23.1 ... 25.35 25.35
    temp_mean           (time, lat, lon) float32 438kB 17.44 17.44 ... 19.07
    precip              (time, lat, lon) float32 438kB 0.0 0.0 0.0 ... 0.0 0.0
    evapotranspiration  (time, lat, lon) float32 438kB 2.426 2.426 ... 2.416


In [7]:
# Install the modern H3 v4 library and data dependencies
!pip install -q h3 xarray netcdf4 pandas numpy

import xarray as xr
import h3
import numpy as np
import pandas as pd
from google.colab import files

# 1. Generate the complete H3 Resolution 8 grid for Nalbari
lat_min, lat_max = 26.14, 26.59
lon_min, lon_max = 91.24, 91.64

# Sample a dense mesh grid (every ~220 meters) to capture all intersecting hexagons
mesh_lats = np.arange(lat_min, lat_max, 0.002)
mesh_lons = np.arange(lon_min, lon_max, 0.002)

hex_set = set()
for lat in mesh_lats:
    for lon in mesh_lons:
        # UPDATED: v4 API call for spatial indexing
        hex_id = h3.latlng_to_cell(lat, lon, 8)
        hex_set.add(hex_id)

hex_ids = list(hex_set)
print(f"Geometry Processing Complete: Identified {len(hex_ids)} unique H3 cells inside Nalbari.")

# 2. Extract precise centroids for the vector grid
# UPDATED: v4 API call for geometric centroids
centroids = [h3.cell_to_latlng(h) for h in hex_ids]
target_lats = [c[0] for c in centroids]
target_lons = [c[1] for c in centroids]

# 3. Load your downloaded data cube
ds = xr.open_dataset("nalbari_weather_cube_2021_2023.nc")

# 4. Map the 3D Data Cube onto the Hexagon Centroids using Bilinear Vector Interpolation
target_lat_da = xr.DataArray(target_lats, dims="hexagon")
target_lon_da = xr.DataArray(target_lons, dims="hexagon")

print("Executing bilinear downscaling across the 3-year temporal stack...")
# FIX: SciPy strictly requires fill_value=None for multi-dimensional out-of-bounds
ds_vectorized = ds.interp(lat=target_lat_da, lon=target_lon_da, method="linear", kwargs={"fill_value": None})

# Border Patch: Fill any NaN values at the absolute edges with the nearest valid data
ds_vectorized = ds_vectorized.bfill(dim='hexagon').ffill(dim='hexagon')
ds_vectorized = ds_vectorized.assign_coords(hexagon=hex_ids)

# 5. Flatten into a highly scannable tidy time-series matrix
print("Data downscaling complete. Formatting final tabular matrix...")
df_matrix = ds_vectorized.to_dataframe().reset_index()

# Sort for time-series integrity per localized zone
df_matrix.sort_values(by=['hexagon', 'time'], inplace=True)

# 6. Compress and export to your local drive
output_file = "nalbari_h3_weather_matrix.csv.gz"
df_matrix.to_csv(output_file, index=False, compression="gzip")
print(f"Success. Vector Matrix saved and compressed: {output_file}")

# Trigger browser download prompt
files.download(output_file)

Geometry Processing Complete: Identified 2671 unique H3 cells inside Nalbari.
Executing bilinear downscaling across the 3-year temporal stack...
Data downscaling complete. Formatting final tabular matrix...
Success. Vector Matrix saved and compressed: nalbari_h3_weather_matrix.csv.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>